# prepare_phase2
Phase 2 Stage 1 CPU coordinator. Run cells 2, 3, and 4 in order. Configure only cell 3.
Uses the pinned Python 3.12 CPU requirements (`requirements-stage2-cpu.txt`), single-threaded BLAS/OpenMP, runs a two-process storage probe under `PHASE1_ROOT/phase2/probes/`, pre-initializes all shared `PHASE1_ROOT/phase2/` parent directories, binds the accepted Phase 1 Stage 4 release, and publishes an immutable `phase2_plan`.


In [ ]:
import os, sys, subprocess
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Research-phase2')
CODE_REF = 'phase2'  # Set to the approved Phase 2 Stage 1 branch or commit SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage 1 settings (edit only this cell)
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
STAGE4_RELEASE_REF = {
    'path': 'results/stage4/releases/20261119T064951Z-07e6f916a67f46fdb88c1812763534bc/_COMPLETE.json',
    'sha256': 'eb5cad38deecdd34e57f88f54a8a4fd8c47d7686fc3d159c240cde657d27c94c',
}
# Optional completed Stage 1 branch refs ({'path': ..., 'sha256': ...} or None)
E07_AUDIT_REF = None
E17_AUTH_REF = None
RUN_VERIFY_STAGE1_BUNDLE = False


In [ ]:
import json

if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 2 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Mounted PHASE1_ROOT not found: {PHASE1_ROOT}')
stage4_marker = PHASE1_ROOT / STAGE4_RELEASE_REF['path']
if not stage4_marker.is_file():
    raise RuntimeError(f'Accepted Stage 4 release manifest not found: {stage4_marker}')

def run_streaming(cmd):
    proc = subprocess.Popen(
        cmd,
        cwd=REPO,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )
    events = []
    assert proc.stdout is not None
    for line in proc.stdout:
        line = line.rstrip('\n')
        if not line:
            continue
        print(line, flush=True)
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    stderr_text = proc.stderr.read().strip() if proc.stderr is not None else ''
    returncode = proc.wait()
    if returncode != 0:
        blockers = [e['blockers'] for e in events if isinstance(e, dict) and 'blockers' in e]
        detail = f'exit={returncode}'
        if blockers:
            detail += f' blockers={blockers[-1]}'
        if stderr_text:
            detail += f' stderr={stderr_text}'
        raise RuntimeError(f'Child command failed: {detail}')
    return events

prepare_cmd = [
    sys.executable, '-u', '-m', 'src.phase2.operations', 'prepare-phase2',
    '--phase1-root', str(PHASE1_ROOT),
    '--storage-scope', 'persistent-drive',
    '--stage4-release', json.dumps(STAGE4_RELEASE_REF),
]
if E07_AUDIT_REF is not None:
    prepare_cmd += ['--e07-audit', json.dumps(E07_AUDIT_REF) if isinstance(E07_AUDIT_REF, dict) else str(E07_AUDIT_REF)]
if E17_AUTH_REF is not None:
    prepare_cmd += ['--e17-auth', json.dumps(E17_AUTH_REF) if isinstance(E17_AUTH_REF, dict) else str(E17_AUTH_REF)]

prepare_events = run_streaming(prepare_cmd)
published = [e for e in prepare_events if e.get('event') == 'phase2_plan_published']
if not published:
    raise RuntimeError('prepare-phase2 did not emit phase2_plan_published event.')
plan_ref = published[-1]['reference']

readback_cmd = [
    sys.executable, '-u', '-m', 'src.phase2.operations', 'readback',
    '--phase1-root', str(PHASE1_ROOT),
    '--manifest', json.dumps(plan_ref),
]
run_streaming(readback_cmd)

if RUN_VERIFY_STAGE1_BUNDLE or (E07_AUDIT_REF is not None and E17_AUTH_REF is not None):
    if E07_AUDIT_REF is None or E17_AUTH_REF is None:
        raise RuntimeError('Both E07_AUDIT_REF and E17_AUTH_REF are required for verify-stage1-bundle.')
    verify_cmd = [
        sys.executable, '-u', '-m', 'src.phase2.operations', 'verify-stage1-bundle',
        '--phase1-root', str(PHASE1_ROOT),
        '--plan', json.dumps(plan_ref),
        '--e07-audit', json.dumps(E07_AUDIT_REF) if isinstance(E07_AUDIT_REF, dict) else str(E07_AUDIT_REF),
        '--e17-auth', json.dumps(E17_AUTH_REF) if isinstance(E17_AUTH_REF, dict) else str(E17_AUTH_REF),
    ]
    run_streaming(verify_cmd)

print('\n=== AUTHORITATIVE PHASE 2 PLAN REFERENCE ===')
print(json.dumps(plan_ref, indent=2))
print('\nREMINDER: Flush and unmount/remount Google Drive (drive.flush_and_unmount()) before launching parallel Stage 2 Colab worker runtimes so all pre-initialized phase2/ parent directories and phase2_plan are visible across runtimes.')
